# Forschungsfrage 1 — Modelle auf Colab laufen lassen

Rechnet Random Forest und XGBoost ungetunt über die Kaggle-Playground-Datensätze
und schreibt die Ergebnisse auf Google Drive.

**Warum alles hier und nicht lokal:** Die Metrikwerte sind bei festem Startwert
auf jedem Rechner gleich, die Zeiten nicht. Damit die Ergebnistabelle am Ende
aus einer Quelle kommt, laufen alle Modelle in derselben Umgebung. Jede
Ergebniszeile hält in der Spalte `umgebung` fest, wo sie entstanden ist — so
fällt auf, wenn doch einmal zwei Rechner gemischt werden.

**Vorher einmalig einrichten:**

1. Kaggle-Token unter [kaggle.com/settings](https://www.kaggle.com/settings) →
   *API* → *Create New Token* holen (beginnt mit `KGAT_`).
2. Links in Colab auf das Schlüssel-Symbol (*Secrets*), neues Secret namens
   `KAGGLE_API_TOKEN` anlegen, Token einfügen, *Notebook access* anschalten.
3. Für die GPU-Modelle später: *Laufzeit* → *Laufzeittyp ändern* → GPU.
   Für Wald und XGBoost reicht die CPU-Laufzeit.

**Nicht alles auf einmal ausführen.** Die Abschnitte brauchen verschiedene
Laufzeiten: 4 und 5 laufen auf CPU, 6 und 7 brauchen eine GPU. TabFM in
Abschnitt 7 stuft `scikit-learn` herunter und darf deshalb erst nach allen
anderen Modellen installiert werden. Abschnitt 9 ist der teuerste Lauf und
steht am Ende. Jede fertige Kombination liegt sofort auf Drive — ein Abbruch
kostet nur den laufenden Datensatz, nicht den Tag.

## 1. Drive einbinden und Ergebnisordner festlegen

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Ordner, in dem die Ergebnis-CSV landet. Aus einem Drive-Link laesst sich der
# Pfad nicht ableiten, der enthaelt nur eine ID - deshalb hier der Name.
ERGEBNIS_ORDNER = Path("/content/drive/MyDrive/Colab Notebooks/output")

if ERGEBNIS_ORDNER.is_dir():
    print("Ergebnisordner gefunden:", ERGEBNIS_ORDNER)
else:
    print("Nicht gefunden:", ERGEBNIS_ORDNER)
    print()
    print("Diese Ordner liegen daneben, den richtigen Namen oben eintragen:")
    eltern = ERGEBNIS_ORDNER.parent
    if eltern.is_dir():
        for p in sorted(eltern.iterdir()):
            if p.is_dir():
                print("   ", p.name)
    else:
        print("auch", eltern, "gibt es nicht - stimmt der Name von MyDrive?")


## 2. Skripte holen

In [ ]:
import os

REPO = "https://github.com/tobiasewald/ProjectBIWise26.git"
ORDNER = Path("/content/ProjectBIWise26")

if ORDNER.is_dir():
    !cd {ORDNER} && git pull --quiet && echo "aktualisiert"
else:
    !git clone --quiet {REPO} {ORDNER} && echo "geklont"

os.chdir(ORDNER)
!ls scripts metadata

Ist das Repository privat, fragt `git clone` nach Zugangsdaten und bleibt
hängen. Dann entweder das Repository öffentlich stellen oder die drei Dateien
`scripts/*.py` plus `metadata/*.csv` von Hand nach Drive legen und von dort
kopieren.

In [ ]:
!pip install --quiet -r requirements.txt
import sklearn, xgboost
print("scikit-learn", sklearn.__version__, "/ xgboost", xgboost.__version__)

Die beiden Versionen müssen `1.6.1` und `3.4.1` sein. Bei Forschungsfrage 1
sind die Voreinstellungen dieser Bibliotheken der Untersuchungsgegenstand —
eine andere Version liefert andere Zahlen. Falls `pip` eine Laufzeit-Neustart
verlangt, einmal neu starten und ab hier weiterlaufen.

## 3. Kaggle-Token aus den Secrets

In [ ]:
from google.colab import userdata

# Das Token wird nur in die Umgebung gesetzt, nie ausgegeben und nie im
# Notebook gespeichert. Das Downloadskript sucht dort zuerst.
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
print("Token gesetzt, Laenge", len(os.environ["KAGGLE_API_TOKEN"]))

## 4. Hauptauswertung: alle Modelle auf 10.000 Zeilen

Jeder Datensatz wird von Kaggle geholt, gerechnet und wieder gelöscht — auf der
Platte liegt nie mehr als einer. Von klein nach groß, damit bei einem Abbruch
schon Ergebnisse da sind.

Fertige Kombinationen werden beim nächsten Lauf übersprungen. Reißt die Sitzung
ab: Zellen 1 bis 3 erneut ausführen, dann diese. Nichts geht verloren.

In [ ]:
!python scripts/run_models.py --laden --daten-loeschen     --daten /content/data     --ausgabe "{ERGEBNIS_ORDNER}"

## 5. Zwischenstand ansehen

In [ ]:
import pandas as pd

erg = pd.read_csv(ERGEBNIS_ORDNER / "model_results.csv", sep=";", encoding="utf-8-sig")
print(len(erg), "Ergebniszeilen,", erg["slug"].nunique(), "Datensaetze")
print("Umgebungen:", erg["umgebung"].unique())
print("Status:", erg["status"].value_counts().to_dict())

erg[erg["status"] == "ok"].pivot_table(
    index="slug", columns="modell", values="wert_mittel"
).join(erg.set_index("slug")[["metrik_code", "richtung"]].drop_duplicates())

## 6. TabPFN

Braucht eine **GPU-Laufzeit** (*Laufzeit* → *Laufzeittyp ändern* → GPU). Auf CPU
verweigert TabPFN-3.5 ab 5.000 Zeilen, unsere Variante trainiert auf 8.000.

Verwendet werden die Gewichte **TabPFN-3.5** (`tabpfn-v3.5-20260909.safetensors`).
Weil der Dateiname das Datum trägt, ist der gemessene Stand zitierbar; er steht
zusätzlich in jeder Ergebniszeile in der Spalte `modell_version`.

**Gerechnet wird lokal auf der Colab-GPU.** Für die neueren Gewichte verlangt
Prior Labs aber einmalig, dass die Lizenz im Konto bestätigt wird, und prüft das
über einen API-Key. Der Key dient also nur der Lizenzprüfung — es werden keine
Daten an Prior Labs geschickt. Das wäre das separate Paket `tabpfn-client`, das
wir nicht verwenden.

Einmalig einzurichten:

1. Auf <https://ux.priorlabs.ai> anmelden oder registrieren.
2. Im Reiter *Licenses* die Lizenz `tabpfn-3-5-license-v1.0` bestätigen.
3. Unter <https://ux.priorlabs.ai/account> den API-Key kopieren.
4. Links im Schlüssel-Symbol (*Secrets*) ein Secret `TABPFN_TOKEN` anlegen, den
   Key eintragen und den Zugriff für dieses Notebook erlauben.

Die Gewichtsdatei selbst liegt offen auf Hugging Face — ein Konto dort ist nicht
nötig.

Ohne Key läuft `--tabpfn-version V2` weiterhin ohne jede Anmeldung; dort gilt auf
CPU allerdings schon ab 1.000 Zeilen eine Grenze.

In [ ]:
!pip install --quiet tabpfn

import os
import torch
from google.colab import userdata

# Ohne gueltigen Token wuerde tabpfn einen Browser oeffnen wollen und auf einen
# Rueckruf an localhost warten. In Colab ist localhost nicht der eigene Rechner,
# das haengt also nur. Mit dieser Variable bricht es sofort mit Anleitung ab.
os.environ["TABPFN_NO_BROWSER"] = "1"

try:
    os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")
    print("Lizenz-Token: aus Colab-Secret TABPFN_TOKEN")
except Exception:
    print("Kein Secret TABPFN_TOKEN gefunden. Entweder Abschnitt 6 einrichten")
    print("oder unten --tabpfn-version V2 anhaengen (laedt ohne Konto).")

print("torch", torch.__version__, "/ GPU verfuegbar:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("ACHTUNG: ohne GPU verweigert TabPFN-3.5 ab 5.000 Zeilen, wir nutzen 8.000.")

In [ ]:
!python scripts/run_models.py --modelle tabpfn --laden --daten-loeschen --daten /content/data --ausgabe "{ERGEBNIS_ORDNER}"

## 7. TabFM

**Wichtig: erst ausführen, wenn Wald, XGBoost und TabPFN fertig sind.**

TabFM bringt eigene Festlegungen für `scikit-learn` und `torch` mit. Unsere
Ergebnisse für Wald und XGBoost entstehen unter scikit-learn 1.6.1 — und bei
Forschungsfrage 1 sind genau die Voreinstellungen dieser Bibliothek der
Untersuchungsgegenstand. Würde die Installation sie
herunterstufen, hätte ein späterer Lauf andere Zahlen, ohne dass es auffällt.

Deshalb: TabFM bekommt eine eigene Laufzeit. *Laufzeit* → *Laufzeit trennen und
löschen*, dann **Abschnitte 1, 2 und 3 erneut** — alle drei. Nach dem Löschen
der Laufzeit ist Drive nicht eingebunden, `ERGEBNIS_ORDNER` nicht gesetzt, die
Skripte sind weg und das Kaggle-Token ist nicht geladen. Fehlt einer der drei,
schreibt der Lauf **keine einzige Zeile**, auch keine Fehlerzeile. Danach hier
weiter. Die Ergebnisdatei wird
nur angehängt, die vorhandenen Zeilen bleiben erhalten. Die Zelle unten prüft danach, ob sich die
Version verschoben hat, und sagt, was für eine Karte zugeteilt wurde.

In [ ]:
!git clone --quiet https://github.com/google-research/tabfm.git /content/tabfm
!cd /content/tabfm && pip install --quiet -e .[pytorch]

import importlib, sklearn, torch
importlib.reload(sklearn)
print("scikit-learn", sklearn.__version__, "/ torch", torch.__version__)
if sklearn.__version__ != "1.6.1":
    print()
    print("ACHTUNG: scikit-learn ist nicht mehr 1.6.1.")
    print("In dieser Laufzeit duerfen rf und xgb nicht mehr gerechnet werden,")
    print("sonst stehen in der Tabelle Zahlen aus zwei Bibliotheksversionen.")

# Welche Karte zugeteilt wurde, entscheidet ueber die Laufzeit von TabFM.
if not torch.cuda.is_available():
    print()
    print("ACHTUNG: keine GPU sichtbar. Laufzeittyp pruefen, sonst rechnet")
    print("TabFM auf der CPU und ein Datensatz dauert Stunden.")
else:
    haupt, neben = torch.cuda.get_device_capability(0)
    gib = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print("GPU: {} | Rechenfaehigkeit {}.{} | {:.1f} GiB".format(
        torch.cuda.get_device_name(0), haupt, neben, gib))
    if haupt < 8:
        print()
        print("Diese Karte hat keine bfloat16-Tensorkerne, die gibt es erst ab 8.0.")
        print("TabFM laedt seine Gewichte in bfloat16 und rechnet hier trotzdem,")
        print("nur um ein Vielfaches langsamer. Gemessen auf einer T4 (7.5):")
        print("2.190 Zeilen in 729 s, hochgerechnet ueber 4 h bei 10.000 Zeilen.")
        print("Erst den Probelauf unten machen, nicht gleich alles starten.")


### 7a. Probelauf: erst messen, dann den ganzen Lauf starten

Zwei Datensätze, einer unter und einer auf der Deckelung. S5E3 ist schon
fertig und wird übersprungen — übrig bleibt genau die Zahl, die
interessiert: **wie lange ein Datensatz mit 10.000 Zeilen braucht**. Mal 33
ist das Budget für den vollen Lauf.

Bei `CUDA out of memory` halbiert das Skript die Bündelgröße selbst und
versucht es erneut, bis hinunter zu 1 — am Ergebnis ändert das nichts, nur
an der Zeit. Mit `--tabfm-batch 4` kann man tiefer anfangen.


In [ ]:
!python scripts/run_models.py --modelle tabfm --only s5e3 s5e7 --laden --daten-loeschen --daten /content/data --ausgabe "{ERGEBNIS_ORDNER}"

### 7b. Voller Lauf

Erst starten, wenn die Zeit aus 7a mal 33 in eine Colab-Sitzung passt. Tut sie
das nicht, ist das eine Entscheidung für die Arbeit und keine technische —
siehe `README.md`, Abschnitt „Was TabFM kostet, gemessen".


In [ ]:
!python scripts/run_models.py --modelle tabfm --laden --daten-loeschen --daten /content/data --ausgabe "{ERGEBNIS_ORDNER}"

## 8. Auswertung: mittlere Ränge

Fasst die Ergebnisse über alle Datensätze zusammen. Zehn verschiedene Metriken
lassen sich nicht mitteln, Ränge schon: Pro Datensatz Rang 1 für das beste
Modell, darüber der Mittelwert. Dazu der Friedman-Test und der Nemenyi-Nachtest
mit kritischer Differenz (Demšar 2006).

Schreibt `ranks_n10000.csv` und das CD-Diagramm `cd_n10000.png` in denselben
Ordner auf Drive.

In [ ]:
!python scripts/aggregate_ranks.py --ergebnisse "{ERGEBNIS_ORDNER}/model_results.csv" --ausgabe "{ERGEBNIS_ORDNER}"

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ERGEBNIS_ORDNER / "cd_n10000.png")))

Im Diagramm gilt: Modelle unter einem gemeinsamen Balken liegen enger
beieinander als die kritische Differenz — ihr Unterschied ist **nicht belegt**.
Ein Vorsprung in der Rangliste allein reicht als Aussage also nicht.

## 9. Zweite Variante: volle Datenmenge

Nur für Wald und Boosting sinnvoll und zeigt, was die Deckelung auf 10.000
Zeilen kostet. Das ist die Sensitivitätsprüfung, nicht das Ergebnis — deshalb
steht sie hier am Ende und nicht vor den Fundamentmodellen.

Dieser Lauf ist der teuerste im ganzen Notebook: S4E7 hat 11,5 Millionen
Zeilen. Eine **CPU-Laufzeit reicht** und ist hier die richtige Wahl — in einer
GPU-Laufzeit würde der Lauf nur Guthaben verbrauchen, ohne die GPU zu nutzen.
Am besten an einem eigenen Tag.

Dass die Zahlen aus einer späteren Sitzung noch zu den übrigen passen, sichert
`requirements.txt`: `scikit-learn` und `xgboost` sind dort exakt festgenagelt,
nicht mit `>=`.


In [ ]:
!python scripts/run_models.py --variante voll --laden --daten-loeschen     --daten /content/data     --ausgabe "{ERGEBNIS_ORDNER}"

Zum Schluss die Ränge für diese Variante. Abschnitt 8 wertet nur `n10000` aus
— das ist die Voreinstellung von `aggregate_ranks.py`. Erst der Lauf hier
unten erzeugt `ranks_voll.csv` und `cd_voll.png` und macht damit sichtbar, was
die Deckelung Wald und Boosting gekostet hat.


In [ ]:
!python scripts/aggregate_ranks.py --variante voll     --ergebnisse "{ERGEBNIS_ORDNER}/model_results.csv"     --ausgabe "{ERGEBNIS_ORDNER}"